
# NeoLM — Train Tokenizer

Train a ByteLevel BPE tokenizer from scratch on the full [NeoCorpus](https://huggingface.co/datasets/MaagDeveloper/NeoCorpus) training split.

## Configuration

- **Vocabulary size:** 49,152
- **Algorithm:** ByteLevel BPE
- **Minimum frequency:** 2
- **Training corpus:** 1,889,952 documents
- **Special tokens:** `<|endoftext|>`, `<|pad|>`

The vocabulary size was selected following the experiments in `01_tokenizer_benchmark.ipynb`.

## Pipeline

1. Load the training corpus.
2. Configure and train the tokenizer.
3. Save the tokenizer in Hugging Face format.
4. Validate encoding, decoding, and special tokens.
5. Analyze token lengths and corpus statistics.


In [ ]:
from pathlib import Path

from datasets import load_from_disk
from tqdm.auto import tqdm

from tokenizers import Tokenizer, Regex, pre_tokenizers
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel, Sequence
from tokenizers.decoders import ByteLevel as ByteLevelDecoder

from transformers import PreTrainedTokenizerFast, AutoTokenizer

DB_DIR = Path("../../artifacts/pretrain_db/final")
MODEL_DIR = Path("../../artifacts/model")

VOCAB_SIZE = 49_152
BATCH_SIZE = 20_000

pretrain_db = load_from_disk(str(DB_DIR))

train_ds = pretrain_db["train"]
val_ds = pretrain_db["validation"]

total_docs = len(train_ds)


def batch_iterator():
    batch = []

    for row in tqdm(
        train_ds,
        total=total_docs,
        desc="Reading tokenizer corpus",
        unit="docs",
    ):
        text = row["text"].strip()

        if not text:
            continue

        batch.append(text)

        if len(batch) >= BATCH_SIZE:
            yield batch
            batch = []

    if batch:
        yield batch


print("=" * 60)
print("NEOLM TOKENIZER TRAINING")
print("=" * 60)
print(f"Train documents:      {total_docs:,}")
print(f"Validation documents: {len(val_ds):,}")
print(f"Target vocabulary:    {VOCAB_SIZE:,}")
print(f"Batch size:           {BATCH_SIZE:,}")
print(f"Output directory:     {MODEL_DIR}")
print("=" * 60)

In [ ]:
import time

SPECIAL_TOKENS = [
    "<|endoftext|>",
    "<|pad|>",
]

PATTERN = (
    r"""'(?i:[sdmt]|ll|ve|re)|"""
    r"""[^\r\n\p{L}\p{N}]?+\p{L}++|"""
    r"""\p{N}{1,3}+|"""
    r""" ?[^\s\p{L}\p{N}]++[\r\n]*+|"""
    r"""\s++$|\s*[\r\n]|\s+(?!\S)|\s"""
)

tokenizer = Tokenizer(BPE(unk_token=None))

tokenizer.pre_tokenizer = Sequence([
    pre_tokenizers.Split(
        Regex(PATTERN),
        behavior="isolated",
        invert=False,
    ),
    ByteLevel(
        add_prefix_space=False,
        use_regex=False,
    ),
])

tokenizer.decoder = ByteLevelDecoder()

trainer = BpeTrainer(
    vocab_size=VOCAB_SIZE,
    min_frequency=2,
    special_tokens=SPECIAL_TOKENS,
    initial_alphabet=ByteLevel.alphabet(),
    show_progress=True,
)

print("=" * 60)
print("NEOLM — FINAL TOKENIZER TRAINING")
print("=" * 60)
print(f"Documents:   {total_docs:,}")
print(f"Target vocab: {VOCAB_SIZE:,}")
print(f"Batch size:   {BATCH_SIZE:,}")
print("=" * 60)

start = time.perf_counter()

tokenizer.train_from_iterator(
    batch_iterator(),
    trainer=trainer,
    length=total_docs,
)

elapsed = time.perf_counter() - start
actual_vocab = tokenizer.get_vocab_size()

print(f"\n{'=' * 60}")
print("TRAINING COMPLETE")
print("=" * 60)
print(f"Actual vocab:   {actual_vocab:,}")
print(f"Training time:  {elapsed / 60:.2f} minutes")
print(f"EOS ID:         {tokenizer.token_to_id('<|endoftext|>')}")
print(f"PAD ID:         {tokenizer.token_to_id('<|pad|>')}")
print("=" * 60)

assert actual_vocab == VOCAB_SIZE
assert tokenizer.token_to_id("<|endoftext|>") == 0
assert tokenizer.token_to_id("<|pad|>") == 1

print("TOKENIZER TRAINING PASSED ✅")

MODEL_DIR.mkdir(parents=True, exist_ok=True)

tokenizer.model.save(str(MODEL_DIR))

print("Vocabulary size:", tokenizer.get_vocab_size())

In [ ]:
pretrained_tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=tokenizer,
    bos_token=None,
    eos_token="<|endoftext|>",
    pad_token="<|pad|>",
)

pretrained_tokenizer.save_pretrained(str(MODEL_DIR))

print("=" * 60)
print("NEOLM TOKENIZER SAVED")
print("=" * 60)
print(f"Directory:   {MODEL_DIR}")
print(f"Vocabulary:  {len(pretrained_tokenizer):,}")
print(f"BOS:         {pretrained_tokenizer.bos_token}")
print(f"EOS:         {pretrained_tokenizer.eos_token_id}")
print(f"PAD:         {pretrained_tokenizer.pad_token_id}")
print("=" * 60)

In [ ]:
# Reload from disk
loaded_tokenizer = AutoTokenizer.from_pretrained(
    str(MODEL_DIR),
    use_fast=True,
    local_files_only=True,
)

assert len(loaded_tokenizer) == VOCAB_SIZE
assert loaded_tokenizer.eos_token_id == 0
assert loaded_tokenizer.pad_token_id == 1

# Round-trip validation
samples = [
    "In the name of Allah.",
    "Hello, world!",
    "Transformers use self-attention mechanisms.",
    "Python, Rust, and C++ are programming languages.",
]

for text in samples:
    ids = loaded_tokenizer.encode(
        text,
        add_special_tokens=False,
    )

    decoded = loaded_tokenizer.decode(
        ids,
        skip_special_tokens=False,
        clean_up_tokenization_spaces=False,
    )

    assert decoded == text, f"Round-trip failed: {text!r}"

    print(f"{text[:45]:45} | {len(ids):3} tokens")

print(f"\n{'=' * 60}")
print("TOKENIZER VALIDATION PASSED")
print(f"{'=' * 60}")

In [ ]:
import numpy as np

CONTEXT_LENGTH = 1024

texts = val_ds["text"]
lengths = []

for i in tqdm(
    range(0, len(texts), 256),
    desc="Tokenizing validation",
):
    batch = texts[i:i + 256]

    encodings = loaded_tokenizer(
        batch,
        add_special_tokens=False,
        truncation=False,
    )

    lengths.extend(
        len(ids) + 1  # EOS per document
        for ids in encodings["input_ids"]
    )

lengths = np.asarray(lengths, dtype=np.int64)

total_tokens = int(lengths.sum())
total_chars = sum(len(t) for t in texts)
total_bytes = sum(len(t.encode("utf-8")) for t in texts)

print(f"\n{'=' * 60}")
print("NEOLM — VALIDATION TOKENIZER STATISTICS")
print(f"{'=' * 60}")

print(f"Documents:           {len(texts):,}")
print(f"Total tokens:        {total_tokens:,}")
print(f"Chars/token:         {total_chars / total_tokens:.3f}")
print(f"Bytes/token:         {total_bytes / total_tokens:.3f}")
print(f"Average tokens/doc:  {lengths.mean():,.1f}")
print(f"Median (P50):        {np.percentile(lengths, 50):,.0f}")
print(f"P95:                 {np.percentile(lengths, 95):,.0f}")
print(f"P99:                 {np.percentile(lengths, 99):,.0f}")
print(f"Maximum:             {lengths.max():,}")

over_context = np.mean(lengths > CONTEXT_LENGTH) * 100

print(f"\nContext length:      {CONTEXT_LENGTH:,}")
print(f"Docs > context:      {over_context:.2f}%")
print(f"{'=' * 60}")